# AI-Generated Image Detector - This notebook provides a quick way to train and test the AI image detector model using the CIFAKE dataset.

In [1]:
# Connect to Google Drive (optional - for saving models)
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Clone the repository

!git clone https://github.com/Marouane-Elgoumiri/ai_image_detector


Cloning into 'ai_image_detector'...
remote: Enumerating objects: 80, done.
remote: Counting objects: 100% (80/80), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 80 (delta 13), reused 74 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (80/80), 496.58 KiB | 15.52 MiB/s, done.
Resolving deltas: 100% (13/13), done.


In [3]:
# Pull latest changes from github repo

%cd /content/ai_image_detector
!git pull

/content/ai_image_detector
Already up to date.


In [1]:
# Install dependencies
%cd /content/ai_image_detector

!pip install -r requirements.txt

/content/ai_image_detector


## Training

### Checking GPU

In [2]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU available: True
GPU: Tesla T4
VRAM: 15.6 GB


### Running train_deep-cnn.py


In [3]:
# test training with 5k/100k training samples

!python train_deep_cnn.py --model efficientnet --batch-size 16 --max-train 5000 --max-val 1000 --epochs 2

GPU Memory: 14.6 GB
END-TO-END DEEP LEARNING TRAINING
Model: efficientnet (small)
Image size: 224
Batch size: 16
Accumulation steps: 2
Effective batch size: 32
Learning rate: 0.0003
Epochs: 2
Loss: label_smooth
Freeze backbone: False
Device: cuda

[1/5] Creating data loaders...
Loading CIFAKE train split...
README.md: 100% 488/488 [00:00<00:00, 2.71MB/s]
data/train-00000-of-00001.parquet: 100% 42.1M/42.1M [00:01<00:00, 39.2MB/s]
data/test-00000-of-00001.parquet: 100% 8.41M/8.41M [00:00<00:00, 20.5MB/s]
Generating train split: 100% 100000/100000 [00:00<00:00, 138398.15 examples/s]
Generating test split: 100% 20000/20000 [00:00<00:00, 129056.50 examples/s]
Loaded 5000 samples for train
Loading CIFAKE test split...
Loaded 1000 samples for test
Train batches: 312
Val batches: 63

[2/5] Creating model...
Downloading: "https://download.pytorch.org/models/efficientnet_v2_s-dd5fe13b.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_s-dd5fe13b.pth
100% 82.7M/82.7M [00:00<00:00, 129MB/s

In [ ]:
# Full training with checkpoint resume
# SESSION 1: Runs fresh, trains until Colab disconnects (~14 epochs)
# SESSION 2+: Resumes from last checkpoint saved to Drive
# Model converges around epoch 11, so 20 epochs is sufficient (not 30)

import os, shutil

DRIVE_CHECKPOINT = '/content/drive/MyDrive/AI_Detector_Models/checkpoint'
os.makedirs(DRIVE_CHECKPOINT, exist_ok=True)

# Check for existing checkpoint on Drive to resume from
resume_flag = ''
if os.path.exists(f'{DRIVE_CHECKPOINT}/best_model.pt'):
    print('Found checkpoint on Drive, resuming training...')
    shutil.copytree(DRIVE_CHECKPOINT, 'models/deep', dirs_exist_ok=True)
    # Show where we left off
    import json
    if os.path.exists(f'models/deep/training_history.json'):
        with open(f'models/deep/training_history.json') as f:
            h = json.load(f)
        print(f'  Previously completed {len(h["val_acc"])} epochs')
        print(f'  Best val acc so far: {max(h["val_acc"]):.2f}%')
    resume_flag = '--resume models/deep/best_model.pt'
else:
    print('No checkpoint found, starting fresh training...')

# Train (resumes automatically if checkpoint exists)
!python train_deep_cnn.py --model efficientnet --batch-size 16 --image-size 224 --epochs 20 --accumulation-steps 2 --early-stopping 10 {resume_flag}

# Save checkpoint to Drive after training (so next session can resume)
if os.path.exists('models/deep/best_model.pt'):
    shutil.copytree('models/deep', DRIVE_CHECKPOINT, dirs_exist_ok=True)
    print(f'\nCheckpoint saved to {DRIVE_CHECKPOINT}')
    print('Next Colab session will resume from here.')

GPU Memory: 14.6 GB
END-TO-END DEEP LEARNING TRAINING
Model: efficientnet (small)
Image size: 224
Batch size: 16
Accumulation steps: 2
Effective batch size: 32
Learning rate: 0.0003
Epochs: 30
Loss: label_smooth
Freeze backbone: False
Device: cuda

[1/5] Creating data loaders...
Loading CIFAKE train split...
Loaded 100000 samples for train
Loading CIFAKE test split...
Loaded 20000 samples for test
Train batches: 6250
Val batches: 1250

[2/5] Creating model...
Total parameters: 20,834,386
Trainable parameters: 20,834,386

[3/5] Setting up trainer...
Training on device: cuda

[4/5] Training...

Starting training for 30 epochs
Model parameters: 20,834,386
Trainable parameters: 20,834,386
------------------------------------------------------------
Epoch 1 [Train]: 100% 6250/6250 [20:56<00:00,  4.98it/s, loss=0.532, acc=81.2]
Epoch 1 [Val]: 100% 1250/1250 [01:11<00:00, 17.36it/s, loss=0.427, acc=91.7]
Epoch 1/30 | Train Loss: 0.5323 | Train Acc: 81.22% | Val Loss: 0.4270 | Val Acc: 91.70% 

### Training Summary & Export

In [ ]:
# Training summary
import json

with open('models/deep/training_history.json') as f:
    h = json.load(f)

best_epoch = h['val_acc'].index(max(h['val_acc'])) + 1
total_epochs = len(h['val_acc'])

print(f'Training completed: {total_epochs} epochs')
print(f'Best val accuracy: {max(h["val_acc"]):.2f}% (epoch {best_epoch})')
print(f'Final train accuracy: {h["train_acc"][-1]:.2f}%')
print(f'Final val accuracy: {h["val_acc"][-1]:.2f}%')
print()

if total_epochs >= 20:
    print('Training complete! Copy model to repo with the cell below.')
else:
    print(f'Training paused at epoch {total_epochs}/20.')
    print('Run the training cell again in a new session to resume.')

In [ ]:
# Copy final model back to repo (run this when training is done)
import os, shutil

repo_models = '/content/ai_image_detector/models/deep'
os.makedirs(repo_models, exist_ok=True)

for fname in ['best_model.pt', 'training_history.json', 'training_summary.json']:
    src = f'models/deep/{fname}'
    dst = f'{repo_models}/{fname}'
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f'Copied {fname}')

print('\nModel ready. Push to GitHub:')
print('  %cd /content/ai_image_detector')
print('  !git add models/')
print('  !git commit -m "Update trained model"')
print('  !git push')

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Run Streamlit App

In [2]:
# Run the Streamlit web app

!streamlit run app.py --server.port 8501 &

/bin/bash: line 1: streamlit: command not found


## Manual Save to Drive (Backup)
The training cell already auto-saves to Drive. Use this only if needed.

In [4]:
# Manual backup: copy model to Google Drive
import shutil, os
drive_path = '/content/drive/MyDrive/AI_Detector_Models/checkpoint'
os.makedirs(drive_path, exist_ok=True)
if os.path.exists('models/deep/best_model.pt'):
    shutil.copytree('models/deep', drive_path, dirs_exist_ok=True)
    print(f'Model saved to {drive_path}')
else:
    print('No model found. Run training first.')